# 30 — B0–B1 paired DEV A/B (official formula, research-only)

Notebook này trả lời câu hỏi hiện tại: **thêm 5.301 keyframe B1 có làm điểm DEV
tăng so với BTC gốc B0 hay không?** Nó dùng cùng code, config, queryset và base
store cho hai nhánh; khác biệt duy nhất là B1 được overlay delta đã khóa bởi
notebook 29.

Trước khi chạy:

1. Add Input đúng Dataset `mfe-data-v2` đã dùng ở notebook 18. Notebook chấp
   nhận layout đã bung hoặc đúng một file `mfe-data-v2.zip`.
2. Bật Internet và ba Secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`,
   `MODAL_TOKEN_SECRET`. `HF_TOKEN` là tùy chọn.
3. Chọn **GPU T4** để nạp/encode query nhanh hơn; không chọn P100 vì PyTorch
   Kaggle hiện tại không hỗ trợ kiến trúc P100. Notebook không chạy lại 5.301 ảnh.
4. Giữ nguyên ô 0 rồi Run All/Save Version một lần.

Notebook chỉ đọc DEV, không mở held-out, không sửa BTC và không merge production.
`DEV_UPLIFT_CONFIRMED` nghĩa là B1 tăng trên bộ DEV với cận dưới bootstrap 95%
không âm; nó chưa phải điểm leaderboard hay quyền tự động đưa vào production.


## 0 — Khóa phép đo


In [ ]:
from pathlib import Path

RUN_SCOPE = "PAIRED_DEV_AB"
CONFIRM_RUN = "B0_B1_DEV"
EXPECTED_B1_COUNT = 5301
EXPECTED_DEV_QUERIES = 63
EXPECTED_METADATA_CORRECTIONS = 33
EXPECTED_NB29_COMPLETE_SHA256 = "d2c38a6a5f7fe810b09d87b6a384b850bcbf942564bf83d910c353eaeafc5fa6"
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 2026
NON_INFERIORITY_MARGIN = 0.0
PROGRESS_EVERY = 5

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/b0-b1-dev-ab")
INPUT = WORK / "input"
BASE_EXTRACT = WORK / "base"
DELTA = WORK / "delta"
OUTPUT = WORK / "output"

if RUN_SCOPE != "PAIRED_DEV_AB" or CONFIRM_RUN != "B0_B1_DEV":
    raise RuntimeError("DEV A/B lock is closed")
if BOOTSTRAP_SAMPLES < 1000:
    raise ValueError("Use at least 1000 paired bootstrap samples")
for path in (WORK, INPUT, BASE_EXTRACT, DELTA, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)


## 1 — Helpers fail-closed


In [ ]:
from __future__ import annotations

import math
from types import SimpleNamespace
from typing import Any


def signed_payload(payload: dict[str, Any], field: str, hash_json) -> dict[str, Any]:
    unsigned = dict(payload)
    claimed = unsigned.pop(field, None)
    if not isinstance(claimed, str) or claimed != hash_json(unsigned):
        raise RuntimeError(f"Invalid signed payload: {field}")
    return unsigned


def snapshot_records(rows: list[dict[str, Any]]) -> list[SimpleNamespace]:
    records = []
    for row in rows:
        frame_idx = int(row["frame_idx"])
        fps = float(row["fps"])
        pts_time = float(row["pts_time"])
        if math.floor(pts_time * fps) != frame_idx:
            raise RuntimeError(f"B1 coordinate does not round-trip: {row['keyframe_id']}")
        shot_ids = list(row.get("shot_ids", []))
        if not shot_ids:
            raise RuntimeError(f"B1 row has no shot id: {row['keyframe_id']}")
        records.append(SimpleNamespace(
            keyframe_id=str(row["keyframe_id"]),
            video_id=str(row["video_id"]),
            frame_idx=frame_idx,
            pts_time=pts_time,
            shot_id=str(shot_ids[0]),
        ))
    ids = [row.keyframe_id for row in records]
    if len(ids) != len(set(ids)):
        raise RuntimeError("B1 snapshot contains duplicate identities")
    return records


def classify_dev_ab(mean_delta: float, lower_95: float, margin: float = 0.0) -> str:
    if mean_delta > 0.0 and lower_95 >= margin:
        return "DEV_UPLIFT_CONFIRMED"
    if mean_delta > 0.0:
        return "DEV_POSITIVE_BUT_UNCERTAIN"
    if mean_delta < 0.0:
        return "DEV_REGRESSION"
    return "NO_MEASURED_CHANGE"


## 2 — Repo, dependencies và research volume


In [ ]:
import gc
import json
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1800:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
try:
    hf_token = secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

askpass = WORK / "git-askpass.sh"
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Restart Session: Kaggle checkout has local changes")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[online]", "modal"])
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import yaml

if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 7:
    raise RuntimeError("GPU này không tương thích PyTorch Kaggle hiện tại; hãy chọn GPU T4, không chọn P100")

from mfe.contracts.ids import pts_time_to_timestamp_ms
from mfe.eval.official import MAX_ANSWERS_PER_QUERY, load_official_queryset, score_one
from mfe.mapping.relational import RelationalMap
from mfe.offline.augmentation.evaluation import VariantRun, paired_bootstrap_interval
from mfe.offline.augmentation.io import canonical_hash, read_jsonl, sha256_file, write_json_atomic, write_jsonl_atomic
from mfe.offline.augmentation.overlay import load_overlay_flat_bundle
from mfe.offline.augmentation.snapshot import build_relational_snapshot
from mfe.offline.bundle import freeze_environment
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.flat_arms import FlatBundle, build_flat_cascade

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
cfg = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
research = ModalVolumeTransport(cfg["output"]["volume"], create_if_missing=False)
root = cfg["output"]["root"].rstrip("/")
nb18_root = f"{root}/gate1-oracle-eval-quang-v1"
nb29_root = f"{root}/gate1-b1-feature-aggregate-v1"
output_root = f"{root}/gate1-b0-b1-dev-ab-v1"
print({
    "repo_commit": repo_commit,
    "scope": RUN_SCOPE,
    "accelerator": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "nb29_complete_sha256": EXPECTED_NB29_COMPLETE_SHA256,
    "heldout_accessed": False,
})


## 3 — Tải và xác minh toàn bộ bundle B1 từ notebook 29


In [ ]:
nb29_complete_path = INPUT / "nb29-COMPLETE.json"
research.get(f"{nb29_root}/COMPLETE.json", nb29_complete_path)
if sha256_file(nb29_complete_path) != EXPECTED_NB29_COMPLETE_SHA256:
    raise RuntimeError("Notebook 29 COMPLETE SHA drift")
nb29 = json.loads(nb29_complete_path.read_text(encoding="utf-8"))
signed_payload(nb29, "manifest_payload_hash", canonical_hash)
if nb29.get("status") != "COMPLETE" or nb29.get("output_status") != "READY_FOR_B0_B1_DEV_AB":
    raise RuntimeError("Notebook 29 is not ready for DEV A/B")
if nb29.get("feature_frames") != EXPECTED_B1_COUNT:
    raise RuntimeError("Notebook 29 B1 count drift")

ledger = nb29.get("artifact_ledger", {})
if canonical_hash(ledger) != nb29.get("artifact_ledger_sha256"):
    raise RuntimeError("Notebook 29 artifact ledger hash drift")
for ordinal, (relative, spec) in enumerate(sorted(ledger.items()), 1):
    remote = f"{nb29_root}/{relative}"
    local = DELTA / relative
    local.parent.mkdir(parents=True, exist_ok=True)
    research.get(remote, local)
    if local.stat().st_size != int(spec["bytes"]):
        raise RuntimeError(f"Notebook 29 artifact size mismatch: {relative}")
    if sha256_file(local) != spec["sha256"] or research.sha256(remote) != spec["sha256"]:
        raise RuntimeError(f"Notebook 29 artifact SHA mismatch: {relative}")
    if ordinal == 1 or ordinal % 5 == 0 or ordinal == len(ledger):
        print({"downloaded_nb29_files": ordinal, "expected": len(ledger)}, flush=True)

aggregate_report = json.loads((DELTA / "aggregate-report.json").read_text(encoding="utf-8"))
signed_payload(aggregate_report, "report_payload_hash", canonical_hash)
if aggregate_report.get("delta_artifact_contract_status") != "PASS":
    raise RuntimeError("Notebook 29 aggregate report is not PASS")
b1_rows = list(read_jsonl(DELTA / "b1-worklist.jsonl"))
b1_ids = [str(row["keyframe_id"]) for row in b1_rows]
if len(b1_ids) != EXPECTED_B1_COUNT or len(set(b1_ids)) != EXPECTED_B1_COUNT:
    raise RuntimeError("Notebook 29 worklist identity/count drift")
delta_bundle = FlatBundle.load(DELTA / "delta-flat")
if delta_bundle.vector_keys.get("siglip") != b1_ids or delta_bundle.vector_keys.get("bge_m3") != b1_ids:
    raise RuntimeError("Notebook 29 vector order drift")
del delta_bundle
gc.collect()
print({"NB29_INPUT": "PASS", "feature_frames": len(b1_ids), "files": len(ledger)})


## 4 — Nhận diện baseline `mfe-data-v2` và khóa lineage notebook 18


In [ ]:
import zipfile

input_root = Path("/kaggle/input")
required_prefix = "stores/flat/data/vectors/"
required_db = "stores/btc_b1/relational/app.db"
expanded_siglip = sorted(input_root.rglob("stores/flat/data/vectors/siglip_vectors.npz"))
expanded_db = sorted(input_root.rglob(required_db))

if len(expanded_siglip) == 1 and len(expanded_db) == 1:
    dataset_root = expanded_siglip[0].parents[4]
    if expanded_db[0] != dataset_root / required_db:
        raise RuntimeError("Baseline flat vectors and app.db are from different inputs")
    BASE_FLAT = dataset_root / "stores/flat"
    BASE_DB = expanded_db[0]
    base_source = {"mode": "expanded_dataset", "root": str(dataset_root)}
else:
    archives = sorted(input_root.rglob("mfe-data-v2.zip"))
    if len(archives) != 1:
        raise RuntimeError(
            "Attach exactly one mfe-data-v2 input containing stores/flat + "
            f"stores/btc_b1 or mfe-data-v2.zip; zip={archives}, siglip={expanded_siglip}, db={expanded_db}"
        )
    archive = archives[0]
    with zipfile.ZipFile(archive) as zipped:
        names = set(zipped.namelist())
        members = sorted(name for name in names if name.startswith(required_prefix) and not name.endswith("/"))
        if "stores/flat/data/vectors/siglip_vectors.npz" not in members or required_db not in names:
            raise RuntimeError("mfe-data-v2.zip is missing flat SigLIP or relational app.db")
        for ordinal, member in enumerate([*members, required_db], 1):
            target = BASE_EXTRACT / member
            info = zipped.getinfo(member)
            if not target.is_file() or target.stat().st_size != info.file_size:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zipped.open(member) as source, target.open("wb") as destination:
                    while chunk := source.read(8 * 1024 * 1024):
                        destination.write(chunk)
            if ordinal == 1 or ordinal % 5 == 0 or ordinal == len(members) + 1:
                print({"extracted_base_files": ordinal, "expected": len(members) + 1}, flush=True)
    BASE_FLAT = BASE_EXTRACT / "stores/flat"
    BASE_DB = BASE_EXTRACT / required_db
    base_source = {"mode": "zip", "archive": str(archive)}

vector_files = sorted((BASE_FLAT / "data/vectors").glob("*"))
if not BASE_DB.is_file() or not vector_files:
    raise RuntimeError("Baseline discovery/extraction incomplete")

nb18_path = INPUT / "nb18-oracle-eval-report.json"
research.get(f"{nb18_root}/oracle-eval-report.json", nb18_path)
nb18 = json.loads(nb18_path.read_text(encoding="utf-8"))
signed_payload(nb18, "report_payload_hash", canonical_hash)
EVAL_CONFIG = REPO / "config/btc_b1.yaml"
QUERYSET = REPO / "eval/queryset_dev.jsonl"
lineage_checks = {
    "base_siglip": sha256_file(BASE_FLAT / "data/vectors/siglip_vectors.npz") == nb18["input_hashes"]["base_siglip"],
    "base_relational_db": sha256_file(BASE_DB) == nb18["input_hashes"]["base_relational_db"],
    "eval_config": sha256_file(EVAL_CONFIG) == nb18["input_hashes"]["eval_config"],
    "dev_queryset": sha256_file(QUERYSET) == nb18["input_hashes"]["dev_queryset"],
}
failed = [name for name, passed in lineage_checks.items() if not passed]
if failed:
    raise RuntimeError(f"Frozen notebook 18 baseline lineage drift: {failed}")
queries = load_official_queryset(QUERYSET)
if len(queries) != EXPECTED_DEV_QUERIES:
    raise RuntimeError(f"DEV query count drift: {len(queries)}")
base_ledger = {
    path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in vector_files
}
print({**base_source, "BASELINE": "PASS", "flat_files": len(vector_files), "queries": len(queries)})


## 5 — Hòa giải metadata đuôi video đối xứng rồi tạo snapshot B1


In [ ]:
import shutil
import sqlite3


snapshot_candidates = snapshot_records(b1_rows)
B0_DB = WORK / "b0-relational" / "app.db"
B0_DB.parent.mkdir(parents=True, exist_ok=True)
B0_DB.unlink(missing_ok=True)
shutil.copy2(BASE_DB, B0_DB)

# A small set of source videos decodes a few seconds beyond the legacy
# total_frames/duration_ms stored in mfe-data-v2. The B1 artifacts prove those
# coordinates exist. Reconcile only metadata in a disposable copy and use that
# same copy for both arms; no baseline keyframe row or score-bearing feature is
# changed.
required_by_video = {}
for item in snapshot_candidates:
    required = required_by_video.setdefault(item.video_id, {"total_frames": 0, "duration_ms": 0})
    required["total_frames"] = max(required["total_frames"], item.frame_idx + 1)
    required["duration_ms"] = max(required["duration_ms"], pts_time_to_timestamp_ms(item.pts_time))

metadata_corrections = []
with sqlite3.connect(B0_DB) as connection:
    with connection:
        for video_id, required in sorted(required_by_video.items()):
            row = connection.execute(
                "SELECT total_frames, duration_ms FROM videos WHERE video_id = ?",
                (video_id,),
            ).fetchone()
            if row is None:
                raise RuntimeError(f"B1 video is absent from baseline SQLite: {video_id}")
            old_total, old_duration = map(int, row)
            new_total = max(old_total, required["total_frames"])
            new_duration = max(old_duration, required["duration_ms"])
            if (new_total, new_duration) != (old_total, old_duration):
                connection.execute(
                    "UPDATE videos SET total_frames = ?, duration_ms = ? WHERE video_id = ?",
                    (new_total, new_duration, video_id),
                )
                metadata_corrections.append({
                    "video_id": video_id,
                    "old_total_frames": old_total,
                    "new_total_frames": new_total,
                    "old_duration_ms": old_duration,
                    "new_duration_ms": new_duration,
                })
    integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
if integrity != "ok":
    raise RuntimeError(f"Reconciled B0 SQLite integrity failed: {integrity}")
if len(metadata_corrections) != EXPECTED_METADATA_CORRECTIONS:
    raise RuntimeError(
        f"Tail metadata correction count drift: {len(metadata_corrections)} != {EXPECTED_METADATA_CORRECTIONS}"
    )

B1_DB = WORK / "b1-relational" / "app.db"
snapshot_report = build_relational_snapshot(
    B0_DB,
    B1_DB,
    snapshot_candidates,
    overwrite=True,
    allow_preexisting_mapping_problems=True,
)
if snapshot_report["delta_rows"] != EXPECTED_B1_COUNT:
    raise RuntimeError("B1 relational snapshot is incomplete")
print({
    "B1_SNAPSHOT": "PASS",
    "delta_rows": snapshot_report["delta_rows"],
    "affected_videos": len(snapshot_report["affected_videos"]),
    "symmetric_metadata_corrections": len(metadata_corrections),
    "baseline_mapping_problems_preserved": True,
})


## 6 — Chạy B0 và B1 trên cùng DEV, có log tiến độ


In [ ]:
import statistics
import time
from functools import lru_cache

import mfe.online.dense_arms as dense_arms
import mfe.online.flat_arms as flat_arms


# Reuse the exact same query vectors in B0 and B1. This both speeds up the
# second arm and removes duplicate encoder work as a source of numeric drift.
_encode_query_uncached = dense_arms.encode_query


@lru_cache(maxsize=None)
def _encode_query_cached(text: str, checkpoint: str, expected_dim: int):
    return _encode_query_uncached(text, checkpoint, expected_dim).copy()


flat_arms.encode_query = _encode_query_cached


def run_variant(name: str, *, relational_db: Path, include_ids: list[str]) -> VariantRun:
    started = time.perf_counter()
    config = __import__("mfe.core.config", fromlist=["load_config"]).load_config(EVAL_CONFIG)
    bundle = load_overlay_flat_bundle(BASE_FLAT, DELTA / "delta-flat", include_ids)
    cascade = build_flat_cascade(config, BASE_FLAT, flat_bundle=bundle)
    relational = RelationalMap(str(relational_db))
    relational.init_schema()
    query_ids, query_types, scores, latencies = [], [], [], []
    top_k = min(config.retrieval.rerank_topk, MAX_ANSWERS_PER_QUERY)
    try:
        for ordinal, query in enumerate(queries, 1):
            query_started = time.perf_counter()
            outcome = score_one(query, cascade, relational, top_k)
            latencies.append((time.perf_counter() - query_started) * 1000.0)
            query_ids.append(query.query_id)
            query_types.append(query.type)
            scores.append(outcome.wide.final_score)
            if ordinal == 1 or ordinal % PROGRESS_EVERY == 0 or ordinal == len(queries):
                elapsed = time.perf_counter() - started
                eta = elapsed / ordinal * (len(queries) - ordinal)
                print({
                    "variant": name,
                    "queries": f"{ordinal}/{len(queries)}",
                    "elapsed_minutes": round(elapsed / 60, 2),
                    "eta_minutes": round(eta / 60, 2),
                }, flush=True)
    finally:
        close = getattr(cascade, "close", None)
        if callable(close):
            close()
        relational.close()
        del cascade, bundle
        gc.collect()
    return VariantRun(
        name=name,
        query_ids=tuple(query_ids),
        query_types=tuple(query_types),
        scores=tuple(scores),
        latency_ms=tuple(latencies),
    )


b0 = run_variant("B0", relational_db=B0_DB, include_ids=[])
b1 = run_variant("B1", relational_db=B1_DB, include_ids=b1_ids)
if b0.query_ids != b1.query_ids or b0.query_types != b1.query_types:
    raise RuntimeError("B0/B1 query identity or order drift")
print({
    "PAIRED_RUN": "COMPLETE",
    "queries": len(b0.query_ids),
    "cached_query_embeddings": _encode_query_cached.cache_info().currsize,
    "B0_mean": b0.mean_score,
    "B1_mean": b1.mean_score,
})


## 7 — Paired bootstrap, kết luận DEV và publish


In [ ]:
interval = paired_bootstrap_interval(
    list(b0.scores),
    list(b1.scores),
    samples=BOOTSTRAP_SAMPLES,
    seed=BOOTSTRAP_SEED,
)
deltas = [right - left for left, right in zip(b0.scores, b1.scores, strict=True)]
rescued = sum(delta > 0 for delta in deltas)
regressed = sum(delta < 0 for delta in deltas)
unchanged = sum(delta == 0 for delta in deltas)
verdict = classify_dev_ab(interval.mean_delta, interval.lower_95, NON_INFERIORITY_MARGIN)
by_type_delta = {
    query_type: b1.by_type[query_type] - b0.by_type[query_type]
    for query_type in sorted(set(b0.by_type) & set(b1.by_type))
}
per_query = [
    {
        "query_id": query_id,
        "query_type": query_type,
        "b0_score": left,
        "b1_score": right,
        "delta": right - left,
        "b0_latency_ms": b0_latency,
        "b1_latency_ms": b1_latency,
    }
    for query_id, query_type, left, right, b0_latency, b1_latency in zip(
        b0.query_ids, b0.query_types, b0.scores, b1.scores, b0.latency_ms, b1.latency_ms, strict=True
    )
]

report = {
    "schema_version": 1,
    "status": "DEV_AB_COMPLETE",
    "verdict": verdict,
    "decision_authority": "paired_dev_diagnostic_only",
    "split": "dev",
    "query_count": len(b0.query_ids),
    "b0_mean": b0.mean_score,
    "b1_mean": b1.mean_score,
    "mean_delta": interval.mean_delta,
    "bootstrap_95": {"lower": interval.lower_95, "upper": interval.upper_95},
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "non_inferiority_margin": NON_INFERIORITY_MARGIN,
    "rescued_queries": rescued,
    "regressed_queries": regressed,
    "unchanged_queries": unchanged,
    "by_type_delta": by_type_delta,
    "latency": {
        "b0_p95_ms": b0.p95_latency_ms,
        "b1_p95_ms": b1.p95_latency_ms,
        "ratio": b1.p95_latency_ms / b0.p95_latency_ms if b0.p95_latency_ms else None,
    },
    "inputs": {
        "nb29_complete_sha256": EXPECTED_NB29_COMPLETE_SHA256,
        "nb29_aggregate_report_sha256": nb29["aggregate_report_sha256"],
        "frozen_base_relational_sha256": sha256_file(BASE_DB),
        "evaluation_b0_relational_sha256": sha256_file(B0_DB),
        "b1_relational_sha256": sha256_file(B1_DB),
        "base_flat_ledger": base_ledger,
        "base_flat_ledger_sha256": canonical_hash(base_ledger),
        "eval_config_sha256": sha256_file(EVAL_CONFIG),
        "dev_queryset_sha256": sha256_file(QUERYSET),
    },
    "snapshot": snapshot_report,
    "symmetric_tail_metadata_corrections": metadata_corrections,
    "repo_commit": repo_commit,
    "environment": freeze_environment(),
    "feature_frames": EXPECTED_B1_COUNT,
    "heldout_accessed": False,
    "btc_mutated": False,
    "official_leaderboard_score_claimed": False,
    "production_go_claimed": False,
    "interpretation": {
        "DEV_UPLIFT_CONFIRMED": "Report the DEV evidence; freeze policy before any one-time held-out gate.",
        "DEV_POSITIVE_BUT_UNCERTAIN": "Positive point estimate, but evidence is not strong enough for a production claim.",
        "NO_MEASURED_CHANGE": "Keep research artifacts; do not integrate B1 into BTC.",
        "DEV_REGRESSION": "Keep research artifacts for diagnosis; do not integrate B1 into BTC.",
    }[verdict],
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}
report["report_payload_hash"] = canonical_hash(report)
report_path = write_json_atomic(OUTPUT / "dev-ab-report.json", report)
per_query_path = write_jsonl_atomic(OUTPUT / "per-query-dev.jsonl", per_query)

artifact_ledger = {
    path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in (report_path, per_query_path)
}
complete = {
    "schema_version": 1,
    "status": "COMPLETE",
    "output_status": verdict,
    "query_count": len(b0.query_ids),
    "b0_mean": b0.mean_score,
    "b1_mean": b1.mean_score,
    "mean_delta": interval.mean_delta,
    "bootstrap_95": {"lower": interval.lower_95, "upper": interval.upper_95},
    "nb29_complete_sha256": EXPECTED_NB29_COMPLETE_SHA256,
    "artifact_ledger": artifact_ledger,
    "artifact_ledger_sha256": canonical_hash(artifact_ledger),
    "heldout_accessed": False,
    "btc_mutated": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}
complete["manifest_payload_hash"] = canonical_hash(complete)
complete_path = write_json_atomic(OUTPUT / "COMPLETE.json", complete)

for path in (report_path, per_query_path):
    remote = f"{output_root}/{path.name}"
    research.put(path, remote)
    if research.sha256(remote) != sha256_file(path):
        raise RuntimeError(f"DEV A/B upload verification failed: {path.name}")

# COMPLETE is the commit marker and must be uploaded last.
complete_remote = f"{output_root}/COMPLETE.json"
research.put(complete_path, complete_remote)
if research.sha256(complete_remote) != sha256_file(complete_path):
    raise RuntimeError("DEV A/B COMPLETE upload verification failed")
print({
    "NOTEBOOK30": "COMPLETE",
    "verdict": verdict,
    "B0_mean": b0.mean_score,
    "B1_mean": b1.mean_score,
    "mean_delta": interval.mean_delta,
    "bootstrap_95": [interval.lower_95, interval.upper_95],
    "rescued": rescued,
    "regressed": regressed,
    "unchanged": unchanged,
    "remote_output": output_root,
    "next": report["interpretation"],
})
